# 07a · Candidatos para el ranking (valid y test)

Etapa 7, paso a (local). El ranking final (§12.4) ordena outfits del clóset de cada usuario. Para medir Recall@K y NDCG@K hace falta, por usuario, una lista de outfits candidatos con la decisión del usuario:

- **valid:** 50 candidatos (los 7 del historial de valid + 43 nuevos). Sirve para ajustar α.
- **test:** 100 candidatos (los 8 del historial de test + 92 nuevos). Aquí se reportan las métricas.

Los nuevos se arman con el clóset y se etiquetan con **el mismo gusto, sesgo y ruido** del usuario (`src/modelo_fly/usuarios.py`, `candidatos`). Ninguno está en el historial del usuario ni en el train de ningún otro (§12.5). Se generan antes de entrenar para que todos los modelos (en Colab) puntúen exactamente los mismos outfits.

**Salida:** `data/processed/hito3/candidatos.csv`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import ndcg_score

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("No se encontró data/raw. Abre el notebook dentro del repositorio.")
sys.path.insert(0, str(ROOT / "src"))

from modelo_fly.usuarios import cargar_items, candidatos, compatibilidad_polyvore

PROC = ROOT / "data" / "processed" / "hito3"
items, atrib, outfits = cargar_items(PROC)
compat, MEDIA, DESV = compatibilidad_polyvore(outfits, atrib)
usuarios = pd.read_csv(PROC / "usuarios.csv")
closets = pd.read_csv(PROC / "closets.csv")
historial = pd.read_csv(PROC / "historial.csv")
print("Usuarios:", len(usuarios), "| interacciones:", len(historial))

Usuarios: 350 | interacciones: 17500


In [2]:
ocupados = {}
for ids, part in zip(historial["item_ids"], historial["particion"]):
    ocupados.setdefault(frozenset(ids.split("|")), set()).add(part)

rng = np.random.default_rng(2027)
tablas = []
for u in usuarios.to_dict("records"):
    c = closets[closets["user_id"] == u["user_id"]]
    h = historial[historial["user_id"] == u["user_id"]]
    for part, n in [("valid", 50), ("test", 100)]:
        tablas.append(candidatos(rng, u, c, atrib, compat, MEDIA, DESV, h, n, part, ocupados))
cand = pd.concat(tablas, ignore_index=True)
cand["n_items"] = cand["item_ids"].str.count(r"\|") + 1
print("Candidatos:", cand.groupby("particion").size().to_dict())
display(cand.groupby(["particion", "origen"])["aceptado"].agg(["size", "mean"]).round(3))
tam = cand.groupby(["user_id", "particion"]).size().unstack()
print("Usuarios con lista incompleta:", int((tam["valid"] < 50).sum()), "/", int((tam["test"] < 100).sum()))

Candidatos: {'test': 35000, 'valid': 17500}


size   mean
particion origen                 
test      historial   2800  0.495
          nuevo      32200  0.496
valid     historial   2450  0.483
          nuevo      15050  0.495

Usuarios con lista incompleta: 0 / 0


In [3]:
# Controles: ningún candidato está en el train de otro usuario ni repetido dentro del usuario.
clave = cand["item_ids"].str.split("|").map(frozenset)
train = {frozenset(i.split("|")) for i in historial.loc[historial["particion"] == "train", "item_ids"]}
nuevos = cand["origen"] == "nuevo"
print("Candidatos nuevos que están en algún train:", int(clave[nuevos].isin(train).sum()))
print("Repetidos dentro de un usuario y partición:",
      int(pd.DataFrame({"u": cand["user_id"], "p": cand["particion"], "k": clave}).duplicated().sum()))
pos = cand.groupby(["user_id", "particion"])["aceptado"].sum().unstack()
print("Aceptados por usuario en test: mediana", int(pos["test"].median()), "| mínimo", int(pos["test"].min()))

# Techo del ranking: ordenar por la probabilidad real de la simulación.
te = cand[cand["particion"] == "test"]
techo = {k: np.mean([ndcg_score([g["aceptado"]], [g["p_real"]], k=k) for _, g in te.groupby("user_id")]) for k in (5, 10)}
azar = {k: np.mean([ndcg_score([g["aceptado"]], [np.random.default_rng(0).random(len(g))], k=k)
                    for _, g in te.groupby("user_id")]) for k in (5, 10)}
print("NDCG@5 / @10 con la probabilidad real (techo):", round(techo[5], 3), "/", round(techo[10], 3))
print("NDCG@5 / @10 al azar:", round(azar[5], 3), "/", round(azar[10], 3))

Candidatos nuevos que están en algún train: 0
Repetidos dentro de un usuario y partición: 0
Aceptados por usuario en test: mediana 49 | mínimo 37


NDCG@5 / @10 con la probabilidad real (techo): 0.919 / 0.905
NDCG@5 / @10 al azar: 0.507 / 0.51


In [4]:
cand.to_csv(PROC / "candidatos.csv", index=False, encoding="utf-8")
print("Guardado: candidatos.csv", cand.shape, (PROC / "candidatos.csv").stat().st_size // 1024, "KB")

Guardado: candidatos.csv (52500, 8) 4494 KB
